# 02 — 1D Parameter Sweep Plots

单参数扫描线图：loss vs wavelength / r / l / d 等，支持 CubicSpline 平滑。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd
from scipy.interpolate import CubicSpline

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'

## 1. 波长扫描 — 多文件叠加

In [ ]:
file_paths = glob.glob(r"D:\仿真\仿真数据\LD-PWB-FA-bridge\FA-PWB-Taper\*.txt", recursive=True)

plt.figure(figsize=(8, 5))

for file in file_paths:
    data = np.loadtxt(file, delimiter=",", skiprows=3)
    wavelength = data[:, 0] * 1e9  # → nm
    t_forward = data[:, 1]
    loss = -np.log10(t_forward) * 10  # → dB
    label = os.path.basename(file).replace(".txt", "")
    plt.plot(wavelength, loss, label=label)

plt.xlabel("Wavelength (nm)", fontsize=14, fontweight='bold')
plt.ylabel("Loss (dB)", fontsize=14, fontweight='bold')
plt.title("Second taper d2: Fundamental TE mode Loss vs. Wavelength", fontsize=16, fontweight='bold')
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.legend(fontsize=12, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

## 2. 波长扫描 — 单文件 + CubicSpline 平滑

In [ ]:
filename = r"D:\simulation\Simulation Project\simulation\LD-PWB-SMF\with ar 1.txt"
data = np.loadtxt(filename, delimiter=',')
wavelengths = data[:, 0] * 1e9
t_forward = data[:, 1]
loss = -np.log10(t_forward) * 10

wavelengths_sorted = wavelengths[::-1]
loss_sorted = loss[::-1]

cs = CubicSpline(wavelengths_sorted, loss_sorted)
x_interp = np.linspace(wavelengths.min(), wavelengths.max(), 200)
y_cs = cs(x_interp)

plt.figure(figsize=(12, 6))
plt.tick_params(axis='both', labelsize=12)
plt.plot(x_interp, y_cs, '-', color='blue', label='T_forward')
plt.xlabel('Wavelength (nm)', fontsize=20, fontweight='bold')
plt.ylabel('Loss (dB)', fontsize=20, fontweight='bold')
plt.title('Fundamental TE mode loss', fontsize=20, fontweight='bold')
plt.grid(True)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.xlim(wavelengths.min() - 5, wavelengths.max() + 5)
plt.ylim(0.7, 0.8)
plt.tight_layout()
plt.show()

## 3. Loss 散点 — 固定 Z 截面

In [ ]:
filename = r"C:\Users\22122\Desktop\专利\z=-9.txt"
data = np.loadtxt(filename, delimiter=',')
wavelengths = data[:, 0]
t_forward = data[:, 1]
loss = -np.log10(t_forward) * 10

plt.figure(figsize=(8, 5))
plt.scatter(wavelengths, loss, color='red', s=60, label='z=-9 μm, 1550 nm')
plt.xlabel('X value (μm)', fontsize=16, fontweight='bold')
plt.ylabel('Loss (dB)', fontsize=16, fontweight='bold')
plt.title('Loss', fontsize=16, fontweight='bold')
plt.grid(True)
plt.legend(fontsize=12)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.tight_layout()
plt.show()

## 4. Loss vs l₁

In [ ]:
data = pd.read_csv(r"d:/simulation/Simulation Project/results/l1_scan/T_forward_sweep_results.txt", sep='\t')
target_x = 16
x = data['l1'].values
y = -10 * np.log10(data['T_forward'].values)

plt.plot(x, y, '-', color='blue', linewidth=2)
plt.scatter(x, y, color='red', s=10, zorder=5, linewidth=2)
plt.axvline(x=target_x, color='red', linestyle='--', alpha=0.5, linewidth=1)
plt.title('Loss', fontsize=16, fontweight='bold')
plt.xlabel(r'$\mathbf{l}_{\boldsymbol{1}}$ (μm)', fontsize=16, fontweight='bold')
plt.ylabel('Loss (dB)', fontsize=16, fontweight='bold')
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Loss vs r₂ (带参数标注)

In [ ]:
data = pd.read_csv(r"d:/simulation/Simulation Project/results/r_2_scan/T_forward_sweep_results.txt", sep='\t')
x = data['r_2'].values
y = -10 * np.log10(data['T_forward'].values)
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)

plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2)
plt.title('Loss', fontsize=16, fontweight='bold')
plt.xlabel(r'$\mathbf{r}_{\boldsymbol{2}}$ (μm)', fontsize=16, fontweight='bold')
plt.ylabel('Loss (dB)', fontsize=16, fontweight='bold')
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.grid(True, alpha=0.3)
plt.text(x=1.5, y=3.6, s=r'$\mathbf{r}_{\boldsymbol{11}}$=0.6 μm', fontsize=16, color='black')
plt.text(x=1.5, y=3.3, s=r'$\mathbf{r}_{\boldsymbol{12}}$=1.5 μm', fontsize=16, color='black')
plt.text(x=1.5, y=3.0, s=r'$\mathbf{l}_{\boldsymbol{1}}$=16 μm', fontsize=16, color='black')
plt.tight_layout()
plt.show()

## 6. T_forward vs l₁（曲线拟合 + 标记目标点）

In [ ]:
data = pd.read_csv(r"d:/simulation/Simulation Project/results/l1_scan/T_forward_sweep_results.txt", sep='\t')
x = data['l1'].values
y = data['T_forward'].values
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)

target_l1 = 16
closest_idx = np.argmin(np.abs(x - target_l1))
target_x = x[closest_idx]
target_y = y[closest_idx]
print(f"Closest point: l1={target_x}, T_forward={target_y:.6f}")

plt.scatter(target_x, target_y, color='red', s=30, zorder=5, label=f'l1={target_x} μm')
plt.axvline(x=target_x, color='red', linestyle='--', alpha=0.5, linewidth=1)
plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2, label='CubicSpline')
plt.title('l1 — T_forward')
plt.xlabel('l1 (μm)')
plt.ylabel('T_forward')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Loss vs r（带参数标注）

In [ ]:
data = pd.read_csv(r"d:/simulation/Simulation Project/results/r_scan_r2=1.1/T_forward_sweep_results.txt", sep='\t')
x = data['r'].values
y = -10 * np.log10(data['T_forward'].values)
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)

plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2)
plt.title('Loss', fontsize=16, fontweight='bold')
plt.xlabel('r (μm)', fontsize=16, fontweight='bold')
plt.ylabel('Loss (dB)', fontsize=16, fontweight='bold')
plt.grid(True, alpha=0.3)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.text(x=102, y=11, s=r'$\mathbf{r}_{\boldsymbol{2}}$=1.1 μm', fontsize=16, color='black')
plt.tight_layout()
plt.show()

## 8. Loss vs l₂

In [ ]:
data = pd.read_csv(r"d:/simulation/Simulation Project/results/l2_scan/T_forward_sweep_results.txt", sep='\t')
x = data['l2'].values
y = -10 * np.log10(data['T_forward'].values) + 0.2
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)

plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2)
plt.title('Loss', fontsize=16, fontweight='bold')
plt.xlabel(r'$\mathbf{l}_{\boldsymbol{2}}$ (μm)', fontsize=16, fontweight='bold')
plt.ylabel('Loss (dB)', fontsize=16, fontweight='bold')
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Loss vs r₃（彩虹渐变曲线 + 空心标记）

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

data = pd.read_csv(r"d:/simulation/Simulation Project/simulation/LD-PWB-SMF/results/r_3_scan/T_forward_sweep_results.txt", sep='\t')
x = data['r_3'].values
y_points = -10 * np.log10(data['T_forward'].values) + 0.27

cs = CubicSpline(x, y_points)
x_interp = np.linspace(x.min(), x.max(), 1000)
y_cs = cs(x_interp)

loss_interp = np.interp(x_interp, x, y_points)
points = np.array([x_interp, y_cs]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

fig, ax = plt.subplots(figsize=(12, 10))
lc = LineCollection(segments, array=loss_interp[:-1], cmap='rainbow_r',
                    norm=Normalize(vmin=loss_interp.min(), vmax=loss_interp.max()),
                    linewidth=2, zorder=2)
ax.add_collection(lc)

ax.scatter(x, y_points, marker='s', s=80, facecolors='none', edgecolors='black',
           linewidths=0.8, zorder=3)

ax.set_xlim(x_interp.min() - 0.2, x_interp.max() + 0.2)
ax.set_ylim(0.36, 0.52)
ax.set_xlabel(r'$\mathbf{r}_{\boldsymbol{3}}$ (μm)', fontsize=30, fontweight='bold')
ax.set_ylabel('Loss (dB)', fontsize=30, fontweight='bold')
for spine in ax.spines.values():
    spine.set_linewidth(2)
ax.grid(True, alpha=0.3)
plt.show()